# Lab 8 - CUDA Programming

## CPU vs GPU Overview

**CPU**
* few cores optimized for *serial processing*
* lower memory bandwith (but direct access to more memory) 
* *latency* optimized cores (faster at a single task but can only perform few at the same time)
* more instructions but slower execution

**GPU**
* hundreds/thousands of smaller, more efficient cores optimized for *multiple tasks simultaneously*
* great for compute-intensive parts of the application
* higher memory bandwidth (but direct access to less memory than CPU)
* *throughput* optimized cores (slower at single tasks but can perform more at the same time)
* limited number of instructions available but faster execution

## CUDA Overview
Here we will outline the main concepts behind the CUDA programming model (focusing on how they are exposed in C). A more in-depth explanation can be found in the [official programming guide](http://docs.nvidia.com/cuda/cuda-c-programming-guide/index.html).

* **host** - CPU and RAM

* **device** - GPU and it's RAM

* **kernels** - special functions, which can be called from host code (regular C code running on the CPU) but are run on the device (GPU) N times in parallel, executed by N CUDA threads.

* **\__global__** - special CUDA C keyword used as part of a method signature to mark that method as a kernel.

* **\__host__** - methods marked with this keyword can be only called from host code and will also run on the host

* **\__device__** - methods marked with this keyword can be only called from device code and will also run on the device

As already mentioned, CUDA and GPUs are highly scalable and parallel thanks to their multithread-first programming model. Kernels are executed N times in separate threads, which are for convenience, grouped in a hierarchy of blocks and grids: 

* **thread block** - CUDA programming model follows a well defined thread hierarchy model in which threads during execution are grouped into so called thread blocks. Thread blocks can be one, two or three dimensional. This very naturally maps to vectors, matrices and volumes.

There's a limit of threads per block since 1) all the threads in a block are expected to reside on the same processor core 2) have to share very limited resources of said core.

We can run a kernel, though, using multiple blocks each with the same number of threads. The total number of threads per kernel in such a case will be number_of_blocks * number_of_threads_per_block.

* **threadIdx.{x,y,z}** - similar to the above these variables identify the thread ID that is being executed with the current block.

* **blockIdx.{x,y,z}** - these 3 special, read-only, variables can be used within CUDA kernels to find out which block is executing the code at a given time. This, and the following read-only variables, are usually used to identify which part of data should be handled by a given kernel instance as there will be N of them running in parallel.

* **blockDim.{x,y,z}** - another special read-only variable accesible within CUDA kernels. Contrary to the previous ones these are constant and describe the total number of threads within a block.

* **grid** - similarly to threads being organized in thread blocks, thread blocks are organized into one, two or three dimensional grids. The number of blocks is dictated by the size of data and/or number of processors available.

* **methodName <<< blocks, threads_per_block>>> (parameters)** - this is how you launch a kernel. You use this syntax in your host code (regular C code which runs on the CPU) but it is executed on the device (GPU). There are numerous variants of this operator but the most basic one takes in 2 numbers: the number of blocks to run on the GPU and the second the number of threads per each block.

Architecting our applications in such a way allows GPUs to take advantage of Streaming Multiprocessors (SMs) which are one of the base elements of modern GPUs (whic are built arround an array of SMs). Scheduler automatically assignes blocks to free SMs as resources become available. Thanks to this your applications will run faster on more powerful GPUs without any changes in code.

# **CUDA Syntax Guide for Lab**

This guide covers essential CUDA concepts, including **memory operations, kernel definitions, shared memory variables, and synchronization**.  

---

## **1. Memory Operations in CUDA**
CUDA uses **global, shared, and local memory**. Below are key memory operations:  

### **Allocating and Copying Memory**
```cpp
int *d_input, *d_output;
int h_input[N] = {1, 2, 3, ..., N};  // Host array

// Allocate memory on the GPU (global memory)
cudaMalloc(&d_input, N * sizeof(int));
cudaMalloc(&d_output, N * sizeof(int));

// Copy data from CPU (host) to GPU (device)
cudaMemcpy(d_input, h_input, N * sizeof(int), cudaMemcpyHostToDevice);

// Copy results back from GPU to CPU
cudaMemcpy(h_input, d_output, N * sizeof(int), cudaMemcpyDeviceToHost);

// Free allocated GPU memory
cudaFree(d_input);
cudaFree(d_output);
```

### **Defining and Launching a Kernel**

```cpp
__global__ void addArrays(int *a, int *b, int *c, int n) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;
    if (idx < n) {
        c[idx] = a[idx] + b[idx];
    }
}

// Launching the kernel (example)
int threadsPerBlock = 256;
int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;
addArrays<<<blocksPerGrid, threadsPerBlock>>>(d_a, d_b, d_c, N);
cudaDeviceSynchronize();  // Ensure completion
```

### **Using shared memory**

```cpp
__global__ void useSharedMemory(int *input, int *output) {
    __shared__ int sharedData[256];  // Shared memory for the block

    int tid = threadIdx.x;
    sharedData[tid] = input[tid];  // Load data from global to shared memory

    __syncthreads();  // Ensure all threads finish writing to shared memory

    output[tid] = sharedData[tid] * 2;  // Example operation: multiply by 2
}
```

### Prerequisite
To execute CUDA code seamlessly in the notebook. We'll use the nvcc4jupyter package. nvcc4jupyter is a Jupyter Notebook plugin that provides cell and line magics to allow running CUDA C++ code from a notebook. This is especially useful when combined with a hosted service such as Kaggle which provide CUDA capable GPUs and you can start learning CUDA C++ without having to install anything or even to own a GPU yourself.

In [ ]:
!pip install nvcc4jupyter

In [ ]:
%load_ext nvcc4jupyter

## Hello, World in CPU

In [ ]:
%%cuda
#include <stdio.h>
    
void hello() {
    printf("Hello, World!\n");
}

int main() {
    hello();
    return 0;
}

## Hello, World in GPU

In [ ]:
%%cuda
#include <stdio.h>

__global__ void hello() {
    printf("Hello, CUDA! Thread [%d] in block [%d]\n", threadIdx.x, blockIdx.x);
}

int main( int argc, char** argv ) {
    hello<<<1,1>>>(); // asynchronous call!
    cudaDeviceSynchronize(); // wait for all operations on the GPU to finish
    return 0;
}

**Try** changing the values (rerun the above cell) between **<<<...>>>** to:

- **<<< 2, 1 >>>**
- **<<< 1, 32 >>>**
- **<<< 2, 16 >>>** 

### Question: What changes do you observe in the code between the CPU and the GPU code?

The `hello` function uses the `__global__` modifier, which tells the compiler it is a CUDA kernel meant to run on the GPU. Inside, it uses `threadIdx.x` and `blockIdx.x` to identify the thread uniquely. Furthermore, the kernel is launched from `main` using the special `<<<1,1>>>` execution configuration syntax instead of a standard function call, and `cudaDeviceSynchronize()` is used to ensure the CPU waits for the GPU to finish execution.

## Task 1a: Adding Arrays
Add two array of arbitrary size > 1000. 

In [ ]:
%%cuda
#include <stdio.h>

__global__ void addArrays(int *a, int *b, int *c, int n) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;
    if (idx < n) {
        c[idx] = a[idx] + b[idx];
    }
}

int main() {
    int n = 1500; // Arr size > 1000
    int size = n * sizeof(int);
    int *h_a, *h_b, *h_c;
    int *d_a, *d_b, *d_c;

    h_a = (int *)malloc(size);
    h_b = (int *)malloc(size);
    h_c = (int *)malloc(size);
    for (int i = 0; i < n; i++) { h_a[i] = i; h_b[i] = i * 2; }

    cudaMalloc(&d_a, size); cudaMalloc(&d_b, size); cudaMalloc(&d_c, size);
    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    int blocksPerGrid = (n + threadsPerBlock - 1) / threadsPerBlock;
    addArrays<<<blocksPerGrid, threadsPerBlock>>>(d_a, d_b, d_c, n);

    cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);
    printf("Result check: h_c[100] = %d (Expected: %d)\n", h_c[100], 100 + 200);

    free(h_a); free(h_b); free(h_c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}


### Task 1b: Adding Arrays
Add two arrays of size 999 where the data is divided between multiple blocks and

1) block= 10 and threads=100
2) block= 10 and threads=10

In [ ]:
%%cuda
#include <stdio.h>

__global__ void addArraysStride(int *a, int *b, int *c, int n) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;
    int stride = blockDim.x * gridDim.x;
    for (int i = idx; i < n; i += stride) {
        c[i] = a[i] + b[i];
    }
}

int main() {
    int n = 999;
    int size = n * sizeof(int);
    int *h_a = (int *)malloc(size);
    int *h_b = (int *)malloc(size);
    int *h_c = (int *)malloc(size);
    for (int i = 0; i < n; i++) { h_a[i] = i; h_b[i] = i; }
    
    int *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, size); cudaMalloc(&d_b, size); cudaMalloc(&d_c, size);
    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);
    
    // 1) block= 10 and threads=100
    addArraysStride<<<10, 100>>>(d_a, d_b, d_c, n);
    cudaDeviceSynchronize();
    
    // 2) block= 10 and threads=10
    addArraysStride<<<10, 10>>>(d_a, d_b, d_c, n);
    cudaDeviceSynchronize();
    
    cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);
    printf("Result check: h_c[998] = %d\n", h_c[998]);
    
    free(h_a); free(h_b); free(h_c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}


### Task 2a: Matrix multiplication
Perform matrix multiplication between two matrices, both of size 100.

In [ ]:
%%cuda
#include <stdio.h>
#define N 100

__global__ void matMul(float *a, float *b, float *c, int n) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < n && col < n) {
        float sum = 0.0f;
        for (int i = 0; i < n; i++) {
            sum += a[row * n + i] * b[i * n + col];
        }
        c[row * n + col] = sum;
    }
}

int main() {
    int size = N * N * sizeof(float);
    float *h_a = (float*)malloc(size);
    float *h_b = (float*)malloc(size);
    float *h_c = (float*)malloc(size);
    for (int i = 0; i < N * N; i++) { h_a[i] = 1.0f; h_b[i] = 2.0f; }

    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, size); cudaMalloc(&d_b, size); cudaMalloc(&d_c, size);
    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(16, 16);
    dim3 blocksPerGrid((N + 15)/16, (N + 15)/16);
    matMul<<<blocksPerGrid, threadsPerBlock>>>(d_a, d_b, d_c, N);

    cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);
    printf("Matrix mult check: C[0][0] = %f, C[99][99] = %f\n", h_c[0], h_c[N*N-1]);

    free(h_a); free(h_b); free(h_c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}


### Task 2b: Matrix multiplication
Perform matrix multiplication between two matrices, where number of rows of one matrix = number of columns of the other matrix = 100. The other dimension can be selected arbitrarily.

In [ ]:
%%cuda
#include <stdio.h>
#define M 50
#define K 100
#define N_DIM 150

__global__ void matMulRect(float *a, float *b, float *c, int m, int k, int n) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < m && col < n) {
        float sum = 0.0f;
        for (int i = 0; i < k; i++) {
            sum += a[row * k + i] * b[i * n + col];
        }
        c[row * n + col] = sum;
    }
}

int main() {
    float *h_a = (float*)malloc(M * K * sizeof(float));
    float *h_b = (float*)malloc(K * N_DIM * sizeof(float));
    float *h_c = (float*)malloc(M * N_DIM * sizeof(float));
    for (int i = 0; i < M * K; i++) h_a[i] = 1.0f;
    for (int i = 0; i < K * N_DIM; i++) h_b[i] = 2.0f;

    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, M * K * sizeof(float));
    cudaMalloc(&d_b, K * N_DIM * sizeof(float));
    cudaMalloc(&d_c, M * N_DIM * sizeof(float));
    cudaMemcpy(d_a, h_a, M * K * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, K * N_DIM * sizeof(float), cudaMemcpyHostToDevice);

    dim3 threads(16, 16);
    dim3 blocks((N_DIM + 15)/16, (M + 15)/16);
    matMulRect<<<blocks, threads>>>(d_a, d_b, d_c, M, K, N_DIM);

    cudaMemcpy(h_c, d_c, M * N_DIM * sizeof(float), cudaMemcpyDeviceToHost);
    printf("Matrix mult rectangular check: C[0][0] = %f, C[49][149] = %f\n", h_c[0], h_c[M*N_DIM-1]);

    free(h_a); free(h_b); free(h_c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}


### Task 3a: 1-D Running Sum

Compute the running sum for window size = 3 assuming that the data is handled by a single block

In [ ]:
%%cuda
#include <stdio.h>

__global__ void movingSumSingleBlock(int *input, int *output, int n) {
    int tid = threadIdx.x;
    if (tid < n - 2) {
        output[tid] = input[tid] + input[tid+1] + input[tid+2];
    }
}

int main() {
    int n = 100;
    int *h_in = (int*)malloc(n * sizeof(int));
    int *h_out = (int*)malloc((n - 2) * sizeof(int));
    for(int i = 0; i < n; i++) h_in[i] = 1;

    int *d_in, *d_out;
    cudaMalloc(&d_in, n * sizeof(int));
    cudaMalloc(&d_out, (n - 2) * sizeof(int));
    cudaMemcpy(d_in, h_in, n * sizeof(int), cudaMemcpyHostToDevice);
    
    // Assumes single block
    movingSumSingleBlock<<<1, 256>>>(d_in, d_out, n);
    
    cudaMemcpy(h_out, d_out, (n - 2) * sizeof(int), cudaMemcpyDeviceToHost);
    printf("Running sum (single block): out[0] = %d, out[97] = %d\n", h_out[0], h_out[n-3]);
    
    free(h_in); free(h_out);
    cudaFree(d_in); cudaFree(d_out);
    return 0;
}


### Task 3b: 1-D Running Sum

Compute the running sum for window size = 3 assuming that the data is handled by multiple blocks

In [ ]:
%%cuda
#include <stdio.h>

__global__ void movingSumMultiBlock(int *input, int *output, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n - 2) {
        output[idx] = input[idx] + input[idx+1] + input[idx+2];
    }
}

int main() {
    int n = 10000;
    int *h_in = (int*)malloc(n * sizeof(int));
    int *h_out = (int*)malloc((n - 2) * sizeof(int));
    for(int i = 0; i < n; i++) h_in[i] = 1;

    int *d_in, *d_out;
    cudaMalloc(&d_in, n * sizeof(int));
    cudaMalloc(&d_out, (n - 2) * sizeof(int));
    cudaMemcpy(d_in, h_in, n * sizeof(int), cudaMemcpyHostToDevice);
    
    int threadsPerBlock = 256;
    int blocks = (n - 2 + threadsPerBlock - 1) / threadsPerBlock;
    movingSumMultiBlock<<<blocks, threadsPerBlock>>>(d_in, d_out, n);
    
    cudaMemcpy(h_out, d_out, (n - 2) * sizeof(int), cudaMemcpyDeviceToHost);
    printf("Running sum (multi block): out[0] = %d, out[9997] = %d\n", h_out[0], h_out[n-3]);
    
    free(h_in); free(h_out);
    cudaFree(d_in); cudaFree(d_out);
    return 0;
}


### Task 4: Maximum value

Determine the maximum value in an array of size 10000 assuming data is split between multiple blocks

In [ ]:
%%cuda
#include <stdio.h>

__global__ void maxReduce(int *input, int *global_max, int n) {
    __shared__ int sdata[256];
    int tid = threadIdx.x;
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    
    sdata[tid] = (i < n) ? input[i] : -2147483648; 
    __syncthreads();
    
    for (int s = blockDim.x / 2; s > 0; s >>= 1) {
        if (tid < s) {
            if (sdata[tid + s] > sdata[tid]) {
                sdata[tid] = sdata[tid + s];
            }
        }
        __syncthreads();
    }
    
    if (tid == 0) {
        atomicMax(global_max, sdata[0]);
    }
}

int main() {
    int n = 10000;
    int *h_in = (int*)malloc(n * sizeof(int));
    for (int i=0; i<n; i++) h_in[i] = i % 1000;
    h_in[423] = 99999;
    
    int h_out = -2147483648;
    int *d_in, *d_out;
    cudaMalloc(&d_in, n * sizeof(int));
    cudaMalloc(&d_out, sizeof(int));
    cudaMemcpy(d_in, h_in, n * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_out, &h_out, sizeof(int), cudaMemcpyHostToDevice);
    
    int threads = 256;
    int blocks = (n + threads - 1) / threads;
    maxReduce<<<blocks, threads>>>(d_in, d_out, n);
    
    cudaMemcpy(&h_out, d_out, sizeof(int), cudaMemcpyDeviceToHost);
    printf("Maximum value is: %d (Expected: 99999)\n", h_out);
    
    free(h_in); cudaFree(d_in); cudaFree(d_out);
    return 0;
}


### Task 5: Sorting
Use your favorite parallelizable algorithm to sort an array of size 10000 such that data is divided between multiple blocks

In [ ]:
%%cuda
#include <stdio.h>

__global__ void oddEvenSort(int *arr, int n, int phase) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (phase == 0) { // Even phase
        if (idx % 2 == 0 && idx < n - 1) {
            if (arr[idx] > arr[idx + 1]) {
                int temp = arr[idx];
                arr[idx] = arr[idx + 1];
                arr[idx + 1] = temp;
            }
        }
    } else { // Odd phase
        if (idx % 2 != 0 && idx < n - 1) {
            if (arr[idx] > arr[idx + 1]) {
                int temp = arr[idx];
                arr[idx] = arr[idx + 1];
                arr[idx + 1] = temp;
            }
        }
    }
}

int main() {
    int n = 10000;
    int *h_in = (int*)malloc(n * sizeof(int));
    for (int i = 0; i < n; i++) h_in[i] = n - i;
    
    int *d_in;
    cudaMalloc(&d_in, n * sizeof(int));
    cudaMemcpy(d_in, h_in, n * sizeof(int), cudaMemcpyHostToDevice);
    
    int threads = 256;
    int blocks = (n + threads - 1) / threads;
    
    for (int i = 0; i < n; i++) {
        oddEvenSort<<<blocks, threads>>>(d_in, n, i % 2);
    }
    
    cudaMemcpy(h_in, d_in, n * sizeof(int), cudaMemcpyDeviceToHost);
    
    bool sorted = true;
    for (int i = 0; i < n - 1; i++) {
        if (h_in[i] > h_in[i+1]) {
            sorted = false;
            break;
        }
    }
    
    if (sorted) printf("Array sorted successfully! First element %d, Last element %d\n", h_in[0], h_in[n-1]);
    else printf("Array not sorted.\n");
    
    free(h_in); cudaFree(d_in);
    return 0;
}


## CUDA supported libraries

### Thrust
The Thrust library is a collection of CUDA-based algorithms and data structures designed for high-performance computing. It provides a rich set of algorithms such as sorting, scan, reduce, and transform, all optimized for CUDA execution. Thrust is similar to the C++ Standard Template Library (STL) but with built-in support for parallel execution on GPUs.

Thrust automatically utilizes CUDA when executing algorithms on GPU data, making it easy for developers to leverage GPU power without manually handling memory management or parallelism.

In [ ]:
%%cuda
#include <thrust/sort.h>
#include <thrust/device_vector.h>
#include <iostream>

int main() {
    thrust::device_vector<int> d_vec = {8, 3, 1, 6, 5, 4, 7, 2};
    
    // Sort the vector using Thrust (GPU-based)
    thrust::sort(d_vec.begin(), d_vec.end());

    // Print sorted vector
    std::cout << "Sorted vector: ";
    for (int i = 0; i < d_vec.size(); i++) {
        std::cout << d_vec[i] << " ";
    }
    std::cout << std::endl;
    return 0;
}


### CuBLAS (CUDA Basic Linear Algebra Subprograms)
CuBLAS is a GPU-accelerated library for performing basic linear algebra operations (e.g., matrix multiplication, addition, etc.). It provides highly optimized versions of these operations, specifically designed to take advantage of the parallel processing capabilities of CUDA-enabled GPUs. CuBLAS is built on top of CUDA and offers efficient implementations of operations like matrix multiplication (gemm), dot product, and transpose.

In [ ]:
%%writefile cublas_example.cu
#include <iostream>
#include <cublas_v2.h>

int main() {
    int N = 3;  // Matrix size
    float A[] = {1, 2, 3, 4, 5, 6, 7, 8, 9}; // 3x3 Matrix A
    float B[] = {9, 8, 7, 6, 5, 4, 3, 2, 1}; // 3x3 Matrix B
    float C[9]; // Resultant 3x3 matrix C

    float *d_A, *d_B, *d_C;
    
    cublasHandle_t handle;
    cublasCreate(&handle);

    // Allocate memory on device
    cudaMalloc(&d_A, N*N * sizeof(float));
    cudaMalloc(&d_B, N*N * sizeof(float));
    cudaMalloc(&d_C, N*N * sizeof(float));

    // Copy data to device
    cudaMemcpy(d_A, A, N*N * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, N*N * sizeof(float), cudaMemcpyHostToDevice);

    // Perform matrix multiplication: C = A * B
    const float alpha = 1.0f;
    const float beta = 0.0f;
    cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N, N, N, N, &alpha, d_A, N, d_B, N, &beta, d_C, N);

    // Copy result back to host
    cudaMemcpy(C, d_C, N*N * sizeof(float), cudaMemcpyDeviceToHost);

    // Print result
    std::cout << "Resulting Matrix C: " << std::endl;
    for (int i = 0; i < N; i++) {
        for (int j = 0; j < N; j++) {
            std::cout << C[i*N + j] << " ";
        }
        std::cout << std::endl;
    }

    // Clean up
    cublasDestroy(handle);
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

In [ ]:
!nvcc -o output cublas_example.cu -lcublas && ./output

### CuPy
CuPy is a GPU-accelerated array library that is highly compatible with NumPy, but it runs on NVIDIA GPUs using CUDA. It allows you to perform operations on large arrays with CUDA support, providing significant performance improvements for numerical computations. CuPy's interface is nearly identical to NumPy, which makes it easy to port CPU-based code to GPU.

In [ ]:
import numpy as np
import cupy as cp
import time

# Define matrix dimensions
N = 1000  # Using a larger matrix to observe performance difference

# Create two random matrices
A_np = np.random.rand(N, N).astype(np.float32)  # NumPy array (CPU)
B_np = np.random.rand(N, N).astype(np.float32)  # NumPy array (CPU)

# CuPy arrays (GPU)
A_cp = cp.random.rand(N, N).astype(cp.float32)  # CuPy array (GPU)
B_cp = cp.random.rand(N, N).astype(cp.float32)  # CuPy array (GPU)

# Matrix multiplication using NumPy (CPU)
start_time = time.time()
C_np = np.matmul(A_np, B_np)  # Matrix multiplication on CPU
end_time = time.time()
cpu_time = end_time - start_time

# Matrix multiplication using CuPy (GPU)
start_time = time.time()
C_cp = cp.matmul(A_cp, B_cp)  # Matrix multiplication on GPU
cp.cuda.Stream.null.synchronize()  # Ensure GPU finishes execution
end_time = time.time()
gpu_time = end_time - start_time

# Print the results
print(f"Matrix multiplication with NumPy (CPU) took: {cpu_time:.6f} seconds")
print(f"Matrix multiplication with CuPy (GPU) took: {gpu_time:.6f} seconds")
